# Shear shape noise: catalog simulation と HT grid covariance の比較

XMM、赤方偏移bin 4、Nside512のshape-noise入りshear共分散を比較するNotebookです。

- 青：3,000回の **catalog-based TreeCorr simulation** の標本共分散
- 橙：100回平均した **grid-based quartic-counts HT** の解析共分散

この2つは異なる推定量です。両者の差をただちに解析式の誤りとは解釈せず、まず全4ブロック（++, +−, −+, −−）とfirst off-diagonalを確認します。simulationの誤差棒は標本共分散のGaussian近似誤差、HTの誤差棒は100 thinning realization間のSEMです。角度は保存値degを表示時だけarcminに変換します。

必要ファイルはサーバー上の以下です。
- `shear_XMM_Nside512_signal_plus_shape_noise_sims/outputs/shear_signal_shape_noise_catalog_3000sims.npz`
- `shear_XMM_Nside512_signal_plus_shape_noise_quartic_counts_down_sampling/outputs/ht_p0.001.npz`

新しく生成した結果を比較する場合は、最初のコードセルの `SIM_FILE` と
`HT_FILE` に両方の絶対パスを指定してください。読み込んだパスは出力されます。
3回のpilotはこの3000回用Notebookでは使用しません。モデルhashの一致確認は維持しています。


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter

# 新しい計算結果を使う場合は、ここに両方の絶対パスを指定してください。
# Noneのままなら従来のフォルダ構成から読み込みます。
SIM_FILE = None
HT_FILE = None

if (SIM_FILE is None) != (HT_FILE is None):
    raise ValueError('SIM_FILE と HT_FILE は両方指定してください。')
if SIM_FILE is None:
    _cwd = Path.cwd().resolve()
    LENSING = next((p for p in (_cwd, *_cwd.parents)
                    if (p / 'shear_XMM_Nside512_signal_plus_shape_noise_sims').is_dir()), None)
    if LENSING is None:
        raise FileNotFoundError('SIM_FILE と HT_FILE に比較するファイルの絶対パスを指定してください。')
    SIM_FILE = LENSING / 'shear_XMM_Nside512_signal_plus_shape_noise_sims/outputs/shear_signal_shape_noise_catalog_3000sims.npz'
    HT_FILE = LENSING / 'shear_XMM_Nside512_signal_plus_shape_noise_quartic_counts_down_sampling/outputs/ht_p0.001.npz'
else:
    SIM_FILE = Path(SIM_FILE).expanduser()
    HT_FILE = Path(HT_FILE).expanduser()
    if not SIM_FILE.is_absolute() or not HT_FILE.is_absolute():
        raise ValueError('SIM_FILE と HT_FILE は絶対パスで指定してください。')
print('Simulation input:', SIM_FILE)
print('HT input:', HT_FILE)
for path in (SIM_FILE, HT_FILE):
    if not path.is_file():
        raise FileNotFoundError(path)

with np.load(SIM_FILE) as f:
    sim = {k: f[k] for k in f.files}
with np.load(HT_FILE) as f:
    ht = {k: f[k] for k in f.files}

C_sim = np.asarray(sim['covariance_total'], dtype=float)
C_ht = np.asarray(ht['covariance_joint'], dtype=float)
E_ht = np.asarray(ht['covariance_sem'], dtype=float)
theta_deg = np.asarray(ht['theta'], dtype=float)
theta_arcmin = 60.0 * theta_deg
n = len(theta_deg)
nsim = int(sim['nsims'])

assert nsim == 3000, 'このNotebookは3000回のsimulationを表示します。pilot結果は指定しないでください。'
assert n == 10 and C_sim.shape == C_ht.shape == E_ht.shape == (2*n, 2*n)
assert int(ht['n_realizations']) == 100
assert np.isclose(float(ht['keep_probability']), 0.001)
assert np.all(np.isfinite(C_sim)) and np.all(np.isfinite(C_ht)) and np.all(np.isfinite(E_ht))
assert str(sim['model_sha256']) == str(ht['model_sha256']), 'simulation とHTのモデルhashが違います'
assert np.allclose(np.asarray(sim['theta']), theta_deg, rtol=0.05), '角度binが一致しません'

# Gaussianベクトルを仮定した、標本共分散要素の標準誤差の目安。
# Cov(Ĉ_ij) ≈ (C_ij^2 + C_ii C_jj)/(Nsim-1)
diag_sim = np.maximum(np.diag(C_sim), 0.0)
E_sim = np.sqrt(np.maximum(C_sim**2 + np.outer(diag_sim, diag_sim), 0.0) / (nsim - 1))

print(f'Nside={int(sim["nside"])}  zbin={int(sim["zbin"])}  galaxies={int(sim["n_galaxies"])}')
print(f'catalog simulations={nsim}; HT thinning realizations={int(ht["n_realizations"])}; p={float(ht["keep_probability"])}')
print('Simulation estimator:', str(sim['estimator']))
print('HT estimator:', str(ht['estimator']))
print('HT noise-added paired difference shape:', np.asarray(ht['covariance_noise_added']).shape)
print('theta [arcmin]:', np.array2string(theta_arcmin, precision=3))


## 1. Covariance各ブロック：offset=0 と first off-diagonal

offset=0では同じ角度bin同士、offset=1では右側（列側）の角度binが1つ大きい要素を描きます。++/−− offset=0は分散です。+−/−+は異なる統計量間の共分散なので、値の符号を保って表示します。

誤差棒は青が3,000 simulationから見積もる標本共分散のGaussian近似誤差、橙がHT 100回平均のSEMです。HTの誤差棒はthinningのばらつきを表し、有限simulation標本誤差とは別の量です。

In [ ]:
def style_cov_axis(ax, values, positive_variance):
    ax.set_xscale('log')
    values = np.asarray(values, dtype=float)
    if positive_variance and np.all(values > 0):
        ax.set_yscale('log')
    else:
        scale = max(float(np.max(np.abs(values))) * 1e-2, 1e-30)
        ax.set_yscale('symlog', linthresh=scale)
    ax.xaxis.set_major_locator(FixedLocator([6, 12, 30, 60, 120, 300]))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x:g}'))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xlabel(r'$\theta$ [arcmin]')
    ax.set_ylabel('Covariance')
    ax.grid(True, which='both', ls='--', alpha=.3)

for offset in (0, 1):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    for x in range(2):
        for y in range(2):
            ax = axes[x, y]
            bins = np.arange(n - offset)
            ii = x*n + bins
            jj = y*n + bins + offset
            vals_sim, vals_ht = C_sim[ii, jj], C_ht[ii, jj]
            positive_variance = (offset == 0 and x == y)
            ax.errorbar(theta_arcmin[bins], vals_sim, yerr=E_sim[ii, jj], fmt='o', ms=4,
                        capsize=2, label='catalog simulation (3000)')
            ax.errorbar(theta_arcmin[bins], vals_ht, yerr=E_ht[ii, jj], fmt='.-', ms=5,
                        capsize=2, label='HT mean (100)')
            style_cov_axis(ax, np.r_[vals_sim, vals_ht], positive_variance)
            signs = ('+' if x == 0 else '-') + ('+' if y == 0 else '-')
            ax.set_title(f'{signs}, offset={offset}')
            ax.legend(fontsize=9)
    plt.show()


## 2. 相関行列と差の全体像

左2枚は各結果の相関行列です。右は `HT − catalog simulation` をsimulation対角成分で規格化した表示です。右図は差の大きさを見るための図であり、有意度ではありません。推定量がgridとcatalogで異なる点も踏まえて読みます。

In [ ]:
def correlation_matrix(C):
    d = np.sqrt(np.maximum(np.diag(C), 0.0))
    den = np.outer(d, d)
    return np.divide(C, den, out=np.full_like(C, np.nan), where=den > 0)

norm = np.sqrt(np.outer(np.maximum(np.diag(C_sim), 0.0), np.maximum(np.diag(C_sim), 0.0)))
D_norm = np.divide(C_ht - C_sim, norm, out=np.full_like(C_sim, np.nan), where=norm > 0)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
for ax, mat, title, lim in zip(
    axes,
    (correlation_matrix(C_sim), correlation_matrix(C_ht), D_norm),
    ('catalog simulation correlation', 'HT correlation', '(HT − simulation) / simulation rms'),
    (1, 1, None),
):
    if lim is None:
        finite = np.abs(mat[np.isfinite(mat)])
        lim = max(float(np.quantile(finite, .98)) if finite.size else 1.0, 1e-12)
    im = ax.imshow(mat, origin='lower', cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')
    ax.axhline(n - .5, color='k', lw=.8); ax.axvline(n - .5, color='k', lw=.8)
    ax.set_title(title); ax.set_xlabel('plus bins, then minus bins'); ax.set_ylabel('plus bins, then minus bins')
    fig.colorbar(im, ax=ax, shrink=.8)
plt.show()


## 3. ノイズ追加分の補助確認

HTファイルの `covariance_noise_added` は、同じthinning seeds/countsを使ったsignal-only HT結果との差です。形状が空でなく、有限な行列なら、paired differenceとして保存されています。このgrid上の増分をcatalog simulationのnoise-only covarianceと同じものとして扱わず、別々に表示します。

In [ ]:
noise_delta = np.asarray(ht['covariance_noise_added'], dtype=float)
if noise_delta.shape == C_ht.shape and np.all(np.isfinite(noise_delta)):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
    for ax, block, title in zip(axes, (noise_delta[:n, :n], noise_delta[n:, n:]),
                                ('HT grid noise increment: ++', 'HT grid noise increment: −−')):
        lim = max(float(np.max(np.abs(block))), 1e-30)
        im = ax.imshow(block, origin='lower', cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')
        ax.set_title(title); ax.set_xlabel('angular bin'); ax.set_ylabel('angular bin')
        fig.colorbar(im, ax=ax, shrink=.8)
    plt.show()
else:
    print('paired signal-only difference is not present; shape =', noise_delta.shape)
